In [1]:
!pip install -q transformers peft trl bitsandbytes datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 15.1 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 29.4 MB/s eta 0:00:00:00:0100:01


In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [3]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

# ---------------------------------------------------
# 1. Model ID (UPDATED)
# ---------------------------------------------------
model_id = "diabolic6045/Sanskrit-qwen-7B-Translate-v2"

# ---------------------------------------------------
# 2. QLoRA Config (KAGGLE SAFE)
# ---------------------------------------------------
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16   # ✅ FIXED (no bf16)
)

# ---------------------------------------------------
# 3. Tokenizer
# ---------------------------------------------------
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Add padding token if missing
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# ---------------------------------------------------
# 4. Model (IMPORTANT FIX)
# ---------------------------------------------------
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map={"": 0}   # ✅ single GPU
)

model = prepare_model_for_kbit_training(model)
model.config.use_cache = False   # ✅ stability


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

...

chat_template.jinja: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [4]:
import os

for root, dirs, files in os.walk('/kaggle'):
    for file in files:
        if "charaka" in file:
            print(os.path.join(root, file))

/kaggle/input/datasets/srisanthdaggumelli/dataset1/train_ready_charaka.jsonl


In [5]:
# ---------------------------------------------------
# 5. LoRA CONFIG (DO NOT APPLY MANUALLY)
# ---------------------------------------------------
peft_config = LoraConfig(
    r=8,   # ✅ reduced for stability
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)

# ❌ DO NOT USE get_peft_model

# ---------------------------------------------------
# 6. Dataset
# ---------------------------------------------------
dataset = load_dataset(
    "json",
    data_files="/kaggle/input/datasets/srisanthdaggumelli/dataset1/train_ready_charaka.jsonl",
    split="train"
)

print(dataset[0])  # sanity check

Generating train split: 0 examples [00:00, ? examples/s]

{'text': "<user>Context: carakasaṃhitā, sūtrasthāna, 1 athāto dīrghaṃjīvitīyam adhyāyaṃ\nPlease explain the following Ayurvedic verse(s):\nvyākhyāsyāmaḥ</user>\n<assistant>According to Chakrapani's Āyurvedadīpikā commentary: guṇatrayavibhedena mūrtitrayam upaiyuṣe / trayībhuve trinetrāya trilokīpataye namaḥ  sarasvatyai namo yasyāḥ prasādāt puṇyakarmabhiḥ / buddhidarpaṇasaṃkrāntaṃ jagadadhyakṣam īkṣyate  brahmadakṣāśvideveśabharadvājapunarvasu  hutāśaveśacarakaprabhṛtibhyo namo namaḥ  pātañjalamahābhāṣyacarakapratisaṃskṛtaiḥ / manovākkāyadoṣāṇāṃ hartre 'hipataye namaḥ  naradattagurūddiṣṭacarakārthānugāminī / kriyate cakradattena ṭīkāyurvedadīpikā  sabhyāḥ sadguruvāksudhāsrutiparisphītaśrutīn asmi vo nālaṃ toṣayituṃ payodapayasā nāmbhonidhis tṛpyati / vyākhyābhāsarasaprakāśanam idaṃ tv asmin yadi prāpyate kvāpi kvāpi kaṇo guṇasya tadasau karṇe kṣaṇaṃ dhīyatām  iha hi dharmārthakāmamokṣaparipanthirogopaśamāya brahmaprabhṛtibhiḥ praṇītāyurvedatantreṣvativistaratvena samprati vartamānālpāy

In [10]:
# ---------------------------------------------------
# 7. Training Config (FIXED)
# ---------------------------------------------------
training_args = SFTConfig(
    output_dir="./charaka_finetuned_model",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="linear",
    num_train_epochs=3,
    logging_steps=10,
    fp16=False,
    bf16=False,
    max_grad_norm=0.0, 
    optim="paged_adamw_8bit",
    save_strategy="epoch",
    report_to="none"
)

# ---------------------------------------------------
# 8. Trainer (FINAL CORRECT VERSION)
# ---------------------------------------------------
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,   # ✅ trainer handles LoRA
    args=training_args,
    processing_class=tokenizer,
    formatting_func=lambda x: x["text"]
) 

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [11]:
# ---------------------------------------------
# 9. Train                                     
# ---------------------------------------------
print("Starting fine-tuning...")              
trainer.train()                               

Starting fine-tuning...


Step,Training Loss
10,3.179452
20,2.628719
30,2.477708
40,2.401542
50,2.281373
60,2.296269
70,2.248323
80,2.106948
90,2.168024
100,2.145306


TrainOutput(global_step=219, training_loss=2.2033815775832086, metrics={'train_runtime': 2415.0422, 'train_samples_per_second': 0.363, 'train_steps_per_second': 0.091, 'total_flos': 1.9075767518674944e+16, 'train_loss': 2.2033815775832086})

In [12]:
trainer.model.save_pretrained("./qwen_charaka_adapter")         
tokenizer.save_pretrained("./qwen_charaka_adapter")             
print("Model fine-tuning complete and saved!")                  

Model fine-tuning complete and saved!


In [20]:
import shutil

# Zip fine-tuned model
shutil.make_archive(
    "qwen_charaka_finetuned",
    'zip',
    "/kaggle/working/charaka_finetuned_model"   # ✅ FIXED PATH
)

# Zip adapter model
shutil.make_archive(
    "qwen_charaka_adapter",
    'zip',
    "/kaggle/working/qwen_charaka_adapter"      # ✅ FIXED PATH
)

print("✅ Zipping complete!")

✅ Zipping complete!


In [21]:
import os

for root, dirs, files in os.walk('/kaggle'):
    for file in files:
        if "charaka" in file:
            print(os.path.join(root, file)) 

/kaggle/input/datasets/srisanthdaggumelli/dataset1/train_ready_charaka.jsonl
/kaggle/working/qwen_charaka_adapter.zip
/kaggle/working/qwen_charaka_finetuned.zip
